# Export & Serialization

This notebook demonstrates HYRR's export and serialization features:

1. **Polars / Pandas DataFrames** — `result_to_polars()`, `result_to_pandas()`
2. **CSV bundle** — `result_to_csv_bundle()` writes a zip with summary + per-layer CSVs
3. **Excel export** — `result_to_excel()`
4. **JSON serialization** — `result_to_json_str()` / `result_from_json_str()` for round-trip results
5. **Config serialization** — `stack_to_config()` / `config_to_json()` for sharing configurations
6. **CLI compare** — compare two saved result files from the command line

## 1. Setup — run a simulation

In [ ]:
from pathlib import Path

from hyrr import (
    Beam, DataStore, Layer, TargetStack,
    compute_stack, result_summary, result_to_polars,
)
from hyrr.materials import resolve_element

db = DataStore("../data/parquet")

beam = Beam(projectile="p", energy_MeV=16.0, current_mA=0.15)
mo100 = resolve_element(db, "Mo", enrichment={100: 1.0})

target = Layer(
    density_g_cm3=10.22,
    elements=[(mo100, 1.0)],
    energy_out_MeV=12.0,
)

stack = TargetStack(
    beam=beam,
    layers=[target],
    irradiation_time_s=86400.0,
    cooling_time_s=86400.0,
)

result = compute_stack(db, stack)
print(f"Produced {sum(len(lr.isotope_results) for lr in result.layer_results)} isotopes")

## 2. Polars DataFrame

In [ ]:
df_pl = result_to_polars(result)
df_pl.sort("activity_Bq", descending=True).head(10)

## 3. Pandas DataFrame

Same data, pandas interface — useful for integration with scikit-learn, seaborn, etc.

In [ ]:
from hyrr import result_to_pandas

df_pd = result_to_pandas(result)
print(f"Type: {type(df_pd).__name__}")
print(f"Columns: {list(df_pd.columns)}")
df_pd.sort_values("activity_Bq", ascending=False).head(10)

## 4. Excel export

In [ ]:
from hyrr import result_to_excel

out_dir = Path("output")
out_dir.mkdir(exist_ok=True)

result_to_excel(result, str(out_dir / "tc99m_results.xlsx"))
print(f"Written: {out_dir / 'tc99m_results.xlsx'}")

## 5. CSV bundle

Produces a zip file with:
- `summary.csv` — one row per isotope per layer
- `layer_0_depth_profile.csv` — depth, energy, dE/dx, heat
- `layer_0_isotopes.csv` — isotope details

In [ ]:
import zipfile

from hyrr import result_to_csv_bundle

bundle_path = str(out_dir / "tc99m_bundle.zip")
result_to_csv_bundle(result, bundle_path)

with zipfile.ZipFile(bundle_path) as zf:
    for name in zf.namelist():
        info = zf.getinfo(name)
        print(f"  {name:<35s}  {info.file_size:>8,} bytes")

## 6. JSON result serialization — round-trip

`result_to_json_str` captures the full simulation output (config + per-layer results + isotope time series). `result_from_json_str` parses it back to a dict for inspection without needing the database.

In [ ]:
from hyrr import result_to_json_str, result_from_json_str, save_result, load_result

# Serialize to JSON string
json_str = result_to_json_str(result)
print(f"JSON size: {len(json_str):,} chars")
print(f"First 200 chars: {json_str[:200]}...")

In [ ]:
# Parse back and inspect
data = result_from_json_str(json_str)

print(f"Keys: {list(data.keys())}")
config = data["config"]
print(f"Beam: {config['beam']}")
print(f"Layers: {len(config['layers'])}")
print(f"Irradiation: {data['irradiation_time_s']} s")

# Inspect isotopes from round-tripped data
layer0 = data["layer_results"][0]["isotope_results"]
for name, iso in list(layer0.items())[:5]:
    print(f"  {iso['name']:<12s}  activity={iso['activity_Bq']:.4E} Bq")

## 7. Save / load result files

Convenient wrappers that write/read JSON to/from disk.

In [ ]:
# Save to file
result_path = str(out_dir / "tc99m_result.json")
save_result(result, result_path)
print(f"Saved: {result_path}")

# Load back
loaded = load_result(result_path)
print(f"Loaded {len(loaded['config']['layers'])} layer(s), "
      f"{len(loaded['layer_results'][0]['isotope_results'])} isotopes in layer 0")

## 8. Config serialization

`stack_to_config` extracts a pure-dict representation of the target stack configuration (beam, layers, enrichments, times). Useful for sharing setups or archiving alongside results.

In [ ]:
import json

from hyrr import stack_to_config, config_to_json, config_from_json

config_dict = stack_to_config(stack)
print(json.dumps(config_dict, indent=2))

In [ ]:
# Round-trip through JSON string
config_json = config_to_json(stack)
config_back = config_from_json(config_json)

print(f"Beam: {config_back['beam']}")
print(f"Layers: {len(config_back['layers'])}")
print(f"Layer 0 elements: {[e['symbol'] for e, _ in config_back['layers'][0]['elements']]}")

## 9. CLI compare

You can compare two saved result files from the command line:

```bash
# Compare full results
hyrr compare result_a.json result_b.json

# Filter to a specific isotope
hyrr compare result_a.json result_b.json --isotope Tc-99m

# Filter to a specific layer
hyrr compare result_a.json result_b.json --layer 0
```

Let's demonstrate by saving two results with different beam energies and comparing them.

In [ ]:
# Run a second simulation at higher energy
beam_18 = Beam(projectile="p", energy_MeV=18.0, current_mA=0.15)
stack_18 = TargetStack(
    beam=beam_18,
    layers=[Layer(
        density_g_cm3=10.22,
        elements=[(mo100, 1.0)],
        energy_out_MeV=12.0,
    )],
    irradiation_time_s=86400.0,
    cooling_time_s=86400.0,
)
result_18 = compute_stack(db, stack_18)

save_result(result, str(out_dir / "result_16MeV.json"))
save_result(result_18, str(out_dir / "result_18MeV.json"))
print("Saved both result files.")

In [ ]:
!cd .. && uv run hyrr compare notebooks/output/result_16MeV.json notebooks/output/result_18MeV.json --isotope Tc-99m

## 10. TOML config for CLI run

The `hyrr run` command accepts a TOML file:

```toml
irradiation_s = 86400
cooling_s = 86400

[beam]
projectile = "p"
energy_MeV = 16.0
current_mA = 0.150

[[layers]]
material = "Mo"
thickness_cm = 0.02
is_monitor = false

[layers.enrichment.Mo]
100 = 0.999
98 = 0.001
```

```bash
hyrr run config.toml --db data/hyrr.sqlite
hyrr run config.toml --db data/hyrr.sqlite --format excel --output-dir results/
hyrr run config.toml --db data/hyrr.sqlite --format both --output-dir results/
```